In [ ]:
import math
import random
import time
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import imageio.v2 as imageio
import os
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

In [ ]:
def generate_circle_points(n, radius=10):
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    return np.column_stack((x, y))


def optimal_path(n):
    return list(range(n)) + [0]


def path_length(points, path):
    total = 0
    for i in range(len(path) - 1):
        p1 = points[path[i]]
        p2 = points[path[i + 1]]
        total += np.linalg.norm(p2 - p1)
    return total


def exact_circle_path_length(n, radius=10):
    # довжина однієї хорди між сусідніми точками
    chord = 2 * radius * np.sin(np.pi / n)
    return n * chord


def create_frames(points, path):
    frames = []

    for k in range(1, len(path)):
        plt.figure()
        plt.scatter(points[:, 0], points[:, 1])

        for i, (x, y) in enumerate(points):
            plt.text(x, y, str(i))

        for i in range(k):
            p1 = points[path[i]]
            p2 = points[path[i + 1]]
            plt.plot([p1[0], p2[0]], [p1[1], p2[1]])

        plt.title(f"Step {k}")

        buf = io.BytesIO()
        plt.savefig(buf, format="png")
        plt.close()
        buf.seek(0)

        frames.append(imageio.imread(buf))

    return frames


def create_gif(frames, output="tsp.gif"):
    imageio.mimsave(output, frames, duration=0.5)
    print(f"Збережено {output}")


if __name__ == "__main__":
    n = 30
    radius = 10

    points = generate_circle_points(n, radius)
    path = optimal_path(n)

    obtained_length = path_length(points, path)
    exact_length = exact_circle_path_length(n, radius)
    error = abs(obtained_length - exact_length)

    print(f"Отримана довжина маршруту: {obtained_length:.6f}")
    print(f"Точна довжина маршруту:    {exact_length:.6f}")
    print(f"Абсолютна похибка:         {error:.6f}")

    frames = create_frames(points, path)
    create_gif(frames)

Отримана довжина маршруту: 62.717078
Точна довжина маршруту:    62.717078
Абсолютна похибка:         0.000000
GIF збережено як tsp.gif


---------

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)


def generate_points_on_circle(n: int, radius: float = 100.0) -> np.ndarray:
    # Генерація n точок на колі
    angles = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    return np.column_stack((x, y)).astype(float)


def theoretical_optimal_length_on_circle(n: int, radius: float) -> float:
    # Теоретично оптимальна довжина маршруту (по колу)
    return float(n * 2.0 * radius * math.sin(math.pi / n))


def euclidean_distance_matrix(points: np.ndarray) -> np.ndarray:
    diff = points[:, None, :] - points[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))


def route_length(route: List[int], dist: np.ndarray) -> float:
    # Обчислення довжини маршруту
    a = np.array(route, dtype=int)
    b = np.array(route[1:] + [route[0]], dtype=int)
    return float(dist[a, b].sum())


def normalize_route(route: List[int]) -> List[int]:
    i = route.index(0)
    rotated = route[i:] + route[:i]

    reversed_route = list(reversed(route))
    j = reversed_route.index(0)
    reversed_rotated = reversed_route[j:] + reversed_route[:j]

    return min(rotated, reversed_rotated)


def thin_history(*arrays, max_frames: int = 80):
    if not arrays:
        return tuple()
    n = len(arrays[0])

    if n <= max_frames:
        return tuple(list(arr) for arr in arrays)

    idx = np.linspace(0, n - 1, max_frames, dtype=int)
    return tuple([arr[i] for i in idx] for arr in arrays)


def draw_point_labels(ax, points: np.ndarray, fontsize: int = 7):
    for idx, (x, y) in enumerate(points):
        ax.annotate(
            str(idx),
            (x, y),
            textcoords="offset points",
            xytext=(4, 4),
            ha="left",
            va="bottom",
            fontsize=fontsize,
            color="darkred",
            zorder=10,
        )


def plot_route(ax, points: np.ndarray, route: List[int], title: str):
    # Побудова маршруту на графіку
    closed = route + [route[0]]
    ordered = points[closed]
    ax.plot(ordered[:, 0], ordered[:, 1], marker="o", markersize=3)
    ax.scatter(points[:, 0], points[:, 1], s=18, c="black", zorder=5)
    draw_point_labels(ax, points, fontsize=7)
    ax.set_title(title)
    ax.axis("equal")
    ax.grid(True, alpha=0.3)


def timed_run(func, *args, **kwargs):
    # Вимірювання часу виконання функції
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return result, elapsed


def relative_error_percent(value: float, reference: float) -> float:
    # Обчислення відносної похибки (%)
    if reference == 0:
        return 0.0
    return abs(value - reference) / reference * 100.0

In [ ]:
# Параметри

class GAParams:
    # Параметри генетичного алгоритму
    def __init__(self, population_size, generations, crossover_rate,
                 mutation_rate, tournament_size, elite_size, no_improve_limit):
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size
        self.elite_size = elite_size
        self.no_improve_limit = no_improve_limit


class ACOParams:
    # Параметри мурашиного алгоритму
    def __init__(self, num_ants, iterations, alpha, beta,
                 evaporation, q, initial_pheromone=1.0, no_improve_limit=25):
        self.num_ants = num_ants
        self.iterations = iterations
        self.alpha = alpha
        self.beta = beta
        self.evaporation = evaporation
        self.q = q
        self.initial_pheromone = initial_pheromone
        self.no_improve_limit = no_improve_limit

# Налаштування GA для різних розмірів задачі
GA_CONFIG = {
    30: dict(population_size=160, generations=400, crossover_rate=0.95, mutation_rate=0.28, tournament_size=5, elite_size=10, no_improve_limit=80),
    50: dict(population_size=220, generations=600, crossover_rate=0.95, mutation_rate=0.25, tournament_size=5, elite_size=12, no_improve_limit=120),
    100: dict(population_size=260, generations=900, crossover_rate=0.94, mutation_rate=0.22, tournament_size=6, elite_size=14, no_improve_limit=200),
    200: dict(population_size=400, generations=1200, crossover_rate=0.92, mutation_rate=0.3, tournament_size=5, elite_size=20, no_improve_limit=300),
}

# Налаштування ACO (кількість мурах = кількість міст)
ACO_CONFIG = {
    30: dict(num_ants=30, iterations=100, alpha=1.0, beta=5.0, evaporation=0.50, q=100.0, initial_pheromone=1.0, no_improve_limit=20),
    50: dict(num_ants=50, iterations=100, alpha=1.0, beta=5.0, evaporation=0.50, q=120.0, initial_pheromone=1.0, no_improve_limit=20),
    100: dict(num_ants=100, iterations=90, alpha=1.0, beta=4.5, evaporation=0.45, q=140.0, initial_pheromone=1.0, no_improve_limit=18),
    200: dict(num_ants=200, iterations=60, alpha=1.0, beta=4.0, evaporation=0.40, q=160.0, initial_pheromone=1.0, no_improve_limit=12),
}


def get_ga_params(n: int) -> GAParams:
    return GAParams(**GA_CONFIG.get(n, GA_CONFIG[200]))


def get_aco_params(n: int) -> ACOParams:
    return ACOParams(**ACO_CONFIG.get(n, ACO_CONFIG[200]))

In [ ]:
# Генетичний алгоритм

def create_random_route(n: int) -> List[int]:
    # Генерація випадкового маршруту
    route = list(range(n))
    random.shuffle(route)
    return route


def tournament_selection(population: List[List[int]], fitness: List[float], k: int) -> List[int]:
    # Вибір найкращого з k випадкових особин
    sampled_idx = random.sample(range(len(population)), k)
    best_idx = min(sampled_idx, key=lambda i: fitness[i])
    return population[best_idx][:]


def ordered_crossover(parent1: List[int], parent2: List[int]) -> List[int]:
    # Кросовер (OX) — збереження порядку міст
    n = len(parent1)
    a, b = sorted(random.sample(range(n), 2))
    child = [-1] * n
    child[a:b + 1] = parent1[a:b + 1]

    fill = [x for x in parent2 if x not in child]
    ptr = 0
    for i in range(n):
        if child[i] == -1:
            child[i] = fill[ptr]
            ptr += 1
    return child


def mutate(route: List[int], rate: float):
    # Мутація маршруту
    if random.random() >= rate:
        return

    i, j = sorted(random.sample(range(len(route)), 2))
    mode = random.choice(("swap", "insert", "invert"))

    if mode == "swap":
        route[i], route[j] = route[j], route[i]
    elif mode == "insert":
        city = route.pop(i)
        route.insert(j, city)
    else:
        route[i:j + 1] = reversed(route[i:j + 1])


def genetic_algorithm_tsp(dist: np.ndarray, params: GAParams):
    # Основний цикл генетичного алгоритму
    n = len(dist)
    population = [create_random_route(n) for _ in range(params.population_size)]

    best_route: Optional[List[int]] = None
    best_length = float("inf")
    best_length_history: List[float] = []
    best_route_history: List[List[int]] = []
    no_improve = 0

    for _ in range(params.generations):
         # Оцінка популяції
        fitness = [route_length(ind, dist) for ind in population]
        # Найкраще рішення покоління
        best_idx = int(np.argmin(fitness))
        gen_best_route = population[best_idx][:]
        gen_best_length = fitness[best_idx]

        # Оновлення глобального найкращого
        if gen_best_length < best_length:
            best_length = gen_best_length
            best_route = gen_best_route[:]
            no_improve = 0
        else:
            no_improve += 1

        best_length_history.append(best_length)
        best_route_history.append(normalize_route(best_route[:]))

        # Елітизм — збереження кращих
        elite_idx = np.argsort(fitness)[:params.elite_size]
        new_population = [population[i][:] for i in elite_idx]

        if best_route is not None:
            new_population[0] = best_route[:]

         # Створення нового покоління
        while len(new_population) < params.population_size:
            p1 = tournament_selection(population, fitness, params.tournament_size)
            p2 = tournament_selection(population, fitness, params.tournament_size)

            if random.random() < params.crossover_rate:
                child1, child2 = ordered_crossover(p1, p2), ordered_crossover(p2, p1)
            else:
                child1, child2 = p1[:], p2[:]

            mutate(child1, params.mutation_rate)
            mutate(child2, params.mutation_rate)

            new_population.append(child1)
            if len(new_population) < params.population_size:
                new_population.append(child2)

        population = new_population

        if no_improve >= params.no_improve_limit:
            break

    return normalize_route(best_route), best_length, best_length_history, best_route_history


In [ ]:
# Мурашиний алгоритм

def choose_next_city(current: int, unvisited: set, pheromone: np.ndarray,
                     heuristic: np.ndarray, alpha: float, beta: float) -> int:
    # Вибір наступного міста на основі феромону та евристики
    candidates = list(unvisited)
    weights = np.array(
        [(pheromone[current, j] ** alpha) * (heuristic[current, j] ** beta) for j in candidates],
        dtype=np.float64
    )

    total = weights.sum()
    if total <= 0 or not np.isfinite(total):
        return random.choice(candidates)

    probabilities = weights / total
    return int(np.random.choice(candidates, p=probabilities))


def build_ant_route(start: int, pheromone: np.ndarray, heuristic: np.ndarray,
                    alpha: float, beta: float) -> List[int]:
    # Побудова маршруту однією мурахою
    n = len(pheromone)
    route = [start]
    unvisited = set(range(n))
    unvisited.remove(start)

    current = start
    while unvisited:
        nxt = choose_next_city(current, unvisited, pheromone, heuristic, alpha, beta)
        route.append(nxt)
        unvisited.remove(nxt)
        current = nxt

    return route


def ant_colony_tsp(dist: np.ndarray, params: ACOParams):
    # Основний цикл мурашиного алгоритму
    n = len(dist)

    pheromone = np.full((n, n), params.initial_pheromone, dtype=np.float64)
    heuristic = np.zeros((n, n), dtype=np.float64)
    mask = dist > 0
    heuristic[mask] = 1.0 / (dist[mask] + 1e-12)

    global_best_route: Optional[List[int]] = None
    global_best_length = float("inf")
    no_improve = 0

    global_best_history: List[float] = []
    iteration_best_history: List[float] = []
    iteration_best_routes_history: List[List[int]] = []
    pheromone_history: List[np.ndarray] = []

    ant_starts = list(range(n))
    if params.num_ants != n:
        ant_starts = [i % n for i in range(params.num_ants)]

    for _ in range(params.iterations):
        # Побудова маршрутів усіма мурахами
        routes: List[List[int]] = []
        lengths: List[float] = []

        for start_city in ant_starts:
            route = build_ant_route(
                start=start_city,
                pheromone=pheromone,
                heuristic=heuristic,
                alpha=params.alpha,
                beta=params.beta
            )
            routes.append(route)
            lengths.append(route_length(route, dist))

        # Пошук найкращого маршруту на поточній ітерації
        best_idx = int(np.argmin(lengths))
        iter_best_route = routes[best_idx][:]
        iter_best_length = lengths[best_idx]

        # Оновлення глобально найкращого розв’язку
        if iter_best_length < global_best_length:
            global_best_length = iter_best_length
            global_best_route = iter_best_route[:]
            no_improve = 0
        else:
            no_improve += 1

        # Випаровування феромону
        pheromone *= (1.0 - params.evaporation)

        # Оновлення феромону на пройдених ребрах
        for route, length in zip(routes, lengths):
            deposit = params.q / max(length, 1e-12)
            for i in range(n):
                a, b = route[i], route[(i + 1) % n]
                pheromone[a, b] += deposit
                pheromone[b, a] += deposit

        global_best_history.append(global_best_length)
        iteration_best_history.append(iter_best_length)
        iteration_best_routes_history.append(normalize_route(iter_best_route[:]))
        pheromone_history.append(pheromone.copy())

        if no_improve >= params.no_improve_limit:
            break

    return (
        normalize_route(global_best_route),
        global_best_length,
        global_best_history,
        iteration_best_history,
        iteration_best_routes_history,
        pheromone_history,
    )



In [ ]:
# Візуалізація

def save_animation(fig, update_func, frames_count: int, save_path: str, interval: int):
    # Створення і збереження GIF-анімації
    if frames_count <= 0:
        plt.close(fig)
        return
    anim = FuncAnimation(fig, update_func, frames=frames_count, interval=interval, repeat=False)
    anim.save(save_path, writer=PillowWriter(fps=max(1, 1000 // interval)))
    plt.close(fig)


def animate_ga(points: np.ndarray, route_history: List[List[int]], length_history: List[float],
               title: str, save_path: str, interval: int = 180):
    # Анімація роботи генетичного алгоритму
    frames_count = min(len(route_history), len(length_history))
    fig, ax = plt.subplots(figsize=(8, 8))

    label_fontsize = 8 if len(points) <= 50 else 6 if len(points) <= 100 else 4

    def update(frame):
        ax.clear()
        route = route_history[frame]
        ordered = points[route + [route[0]]]

        ax.scatter(points[:, 0], points[:, 1], s=18, c="black", zorder=5)
        draw_point_labels(ax, points, fontsize=label_fontsize)
        ax.plot(ordered[:, 0], ordered[:, 1], linewidth=2.0, color="royalblue")
        ax.set_title(
            f"{title}\nПокоління: {frame + 1}/{frames_count} | "
            f"Найкраща довжина: {length_history[frame]:.2f}"
        )
        ax.axis("equal")
        ax.grid(True, alpha=0.3)

    save_animation(fig, update, frames_count, save_path, interval)


def draw_pheromone_edges(ax, points: np.ndarray, pheromone: np.ndarray, top_k_edges: int = 250):
    # Відображення найсильніших феромонних ребер
    n = len(points)
    edges = [(i, j, pheromone[i, j]) for i in range(n) for j in range(i + 1, n)]
    edges.sort(key=lambda x: x[2], reverse=True)
    edges = edges[:top_k_edges]

    if not edges:
        return

    values = np.array([e[2] for e in edges], dtype=np.float64)
    vmin, vmax = values.min(), values.max()

    if vmax - vmin < 1e-12:
        normalized = np.full_like(values, 0.5)
    else:
        normalized = (values - vmin) / (vmax - vmin)

    cmap = matplotlib.colormaps["autumn"]

    for (i, j, _), t in zip(edges, normalized):
        ax.plot(
            [points[i, 0], points[j, 0]],
            [points[i, 1], points[j, 1]],
            color=cmap(t),
            alpha=0.4 + 0.6 * t,
            linewidth=1.5 + 5.5 * t,
        )


def animate_aco(points: np.ndarray, route_history: List[List[int]],
                iteration_best_lengths: List[float], pheromone_history: List[np.ndarray],
                title: str, save_path: str, interval: int = 220):
    # Анімація роботи мурашиного алгоритму
    frames_count = min(len(route_history), len(iteration_best_lengths), len(pheromone_history))
    fig, ax = plt.subplots(figsize=(8, 8))

    n = len(points)
    top_k_edges = n * 2 if n <= 50 else n
    label_fontsize = 8 if n <= 50 else 6 if n <= 100 else 4

    def update(frame):
        ax.clear()
        current_route = route_history[frame]
        current_pheromone = pheromone_history[frame]

        draw_pheromone_edges(ax, points, current_pheromone, top_k_edges=top_k_edges)
        ax.scatter(points[:, 0], points[:, 1], s=18, c="black", zorder=5)
        draw_point_labels(ax, points, fontsize=label_fontsize)

        ordered = points[current_route + [current_route[0]]]
        ax.plot(
            ordered[:, 0], ordered[:, 1],
            color="deepskyblue", linewidth=1.5, marker="o", markersize=2.8,
        )

        ax.set_title(
            f"{title}\nІтерація: {frame + 1}/{frames_count} | "
            f"Краща довжина ітерації: {iteration_best_lengths[frame]:.2f}"
        )
        ax.axis("equal")
        ax.grid(True, alpha=0.3)

    save_animation(fig, update, frames_count, save_path, interval)


def save_static_figure(points: np.ndarray, ga_route: List[int], ga_history: List[float],
                       aco_route: List[int], aco_global_history: List[float],
                       aco_iter_history: List[float], exact_length: float,
                       n: int, output_png: str):
     # Побудова та збереження порівняльного графіка (GA vs ACO)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    plot_route(axes[0, 0], points, ga_route, f"GA маршрут, n={n}, L={ga_history[-1]:.2f}")

    axes[0, 1].plot(ga_history, label="GA best")
    axes[0, 1].axhline(exact_length, linestyle="--", label="Еталон")
    axes[0, 1].set_title(f"GA збіжність, n={n}")
    axes[0, 1].set_xlabel("Покоління")
    axes[0, 1].set_ylabel("Найкраща довжина")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    plot_route(axes[1, 0], points, aco_route, f"ACO маршрут, n={n}, L={aco_global_history[-1]:.2f}")

    axes[1, 1].plot(aco_global_history, label="Глобально кращий")
    axes[1, 1].plot(aco_iter_history, label="Кращий на ітерації", alpha=0.8)
    axes[1, 1].axhline(exact_length, linestyle="--", label="Еталон")
    axes[1, 1].set_title(f"ACO збіжність, n={n}")
    axes[1, 1].set_xlabel("Ітерація")
    axes[1, 1].set_ylabel("Довжина")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_png, dpi=160, bbox_inches="tight")
    plt.close(fig)



# Вивід порівняння результатів алгоритмів
def print_experiment_table(n: int, exact_length: float,
                           ga_best: float, ga_time: float,
                           aco_best: float, aco_time: float):
    ga_err = relative_error_percent(ga_best, exact_length)
    aco_err = relative_error_percent(aco_best, exact_length)

    print(f"\n\nКількість вершин: {n}")
    print(f"Очікувана довжина: {exact_length:.4f}")
    print(f"{'№':<3} {'Алгоритм':<15} {'Найкраща довжина':<22} {'Похибка, %':<15} {'Час виконання':<15}")
    print("-" * 70)
    print(f"{'1.':<3} {'Генетичний':<15} {ga_best:<22.4f} {ga_err:<15.6f} {f'{ga_time:.2f} с':<15}")
    print(f"{'2.':<3} {'Мурашиний':<15} {aco_best:<22.4f} {aco_err:<15.6f} {f'{aco_time:.2f} с':<15}")
    print("-" * 70)


In [ ]:
# Експеримент

def run_experiment_for_n(n: int, radius: float = 100.0, save_gif: bool = True, save_png: bool = True):
    points = generate_points_on_circle(n=n, radius=radius)
    dist = euclidean_distance_matrix(points)
    exact_length = theoretical_optimal_length_on_circle(n=n, radius=radius)

     # Отримуємо параметри для GA залежно від n
    ga_params = get_ga_params(n)
    (ga_route, ga_best, ga_history, ga_route_history), ga_time = timed_run(
        genetic_algorithm_tsp, dist, ga_params
    )

     # Отримуємо параметри для ACO
    aco_params = get_aco_params(n)
    (
        aco_route,
        aco_best,
        aco_global_history,
        aco_iter_history,
        aco_route_history,
        aco_pheromone_history,
    ), aco_time = timed_run(ant_colony_tsp, dist, aco_params)

    print_experiment_table(n, exact_length, ga_best, ga_time, aco_best, aco_time)

    # Збереження графіків
    if save_png:
        save_static_figure(
            points=points,
            ga_route=ga_route,
            ga_history=ga_history,
            aco_route=aco_route,
            aco_global_history=aco_global_history,
            aco_iter_history=aco_iter_history,
            exact_length=exact_length,
            n=n,
            output_png=f"comparison_n{n}.png",
        )

    if save_gif:
        ga_route_anim, ga_hist_anim = thin_history(ga_route_history, ga_history, max_frames=80)
        aco_route_anim, aco_iter_anim, aco_pher_anim = thin_history(
            aco_route_history, aco_iter_history, aco_pheromone_history, max_frames=80
        )

        animate_ga(
            points=points,
            route_history=ga_route_anim,
            length_history=ga_hist_anim,
            title=f"Генетичний алгоритм, n={n}",
            save_path=f"ga_n{n}.gif",
            interval=180,
        )

        animate_aco(
            points=points,
            route_history=aco_route_anim,
            iteration_best_lengths=aco_iter_anim,
            pheromone_history=aco_pher_anim,
            title=f"Мурашиний алгоритм, n={n}",
            save_path=f"aco_n{n}.gif",
            interval=220,
        )




def main():
    set_seed(42)

    sizes = [30, 50, 100, 200]
    radius = 100.0

    for n in sizes:
        run_experiment_for_n(n=n, radius=radius, save_gif=True, save_png=True)


if __name__ == "__main__":
    main()



Кількість вершин: 30
----------------------------------------------------------------------
Очікувана довжина: 627.1708
----------------------------------------------------------------------
№   Алгоритм        Найкраща довжина       Похибка, %      Час виконання  
----------------------------------------------------------------------
1.  Генетичний      627.1708               0.000000        1.11 с         
2.  Мурашиний       627.1708               0.000000        1.36 с         
----------------------------------------------------------------------

Кількість вершин: 50
----------------------------------------------------------------------
Очікувана довжина: 627.9052
----------------------------------------------------------------------
№   Алгоритм        Найкраща довжина       Похибка, %      Час виконання  
----------------------------------------------------------------------
1.  Генетичний      627.9052               0.000000        4.90 с         
2.  Мурашиний       627.9052